# IMDb Sentiment Analysis
Transforms text into a numerical matrix using TF-IDF Vectorization and trains a random forest classifier.

In [0]:
import mlflow
import mlflow.spark
from mlflow.models import infer_signature

from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StopWordsRemover
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [0]:
# Load the data into Spark DataFrames
spark_train_df = spark.table("imdb_sentiment_train")
spark_test_df = spark.table("imdb_sentiment_test")

print(f"Dataset successfully loaded across distributed cluster partitions.")


In [0]:
# Set up preprocessing and model pipeline

tokenizer = Tokenizer(inputCol="text", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
    
hashing_tf = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=5000)
idf = IDF(inputCol="raw_features", outputCol="features")
    
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100, maxDepth=10, seed=42)
    
pipeline = Pipeline(stages=[tokenizer, remover, hashing_tf, idf, rf])


In [0]:
# auto-logging is not supported on serverless
#mlflow.spark.autolog()

from mlflow.models import infer_signature

run_name = "imdb_sparkml_run"

with mlflow.start_run(run_name=run_name) as run:
    # Capture the active run information
    active_run_id = run.info.run_id
    print(f"Active Run ID: {active_run_id}")

    # Train the model
    model = pipeline.fit(spark_train_df)
    
    # Generate predictions for custom evaluations
    predictions = model.transform(spark_test_df)
    
    # Step 4: Evaluate pipeline performance
    evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    accuracy = evaluator_acc.evaluate(predictions)
    print(f"🎯 Distributed SparkML Model Accuracy: {accuracy:.4f}")
    
    mlflow.log_metric("test_accuracy", accuracy)
    
    # Log the complete transformer pipeline model artifact

    # Unity Catalog requires a model signature; infer it from a sample of raw text and the pipeline's predictions
    sample_input = spark_test_df.select("text").limit(5)
    sample_predictions = model.transform(sample_input).select("prediction")
    signature = infer_signature(sample_input.toPandas(), sample_predictions.toPandas())
    mlflow.spark.log_model(
        model,
        "model",
        signature=signature,
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp"
    )

In [0]:
# 1. Register the SparkML model
mlflow.register_model(
    model_uri=f"runs:/{active_run_id}/model",
    name="imdb_sparkml_model"
)